##Daily Fortune 500 Stock Updates

In [2]:
import requests
import pandas as pd
from bs4 import BeautifulSoup
from collections import OrderedDict
import json

In [3]:
soup = BeautifulSoup(requests.get("https://stockanalysis.com/list/biggest-companies/").content, "html.parser")

In [11]:
##Scrap table with company data from site and store it into a list of dictionaries

def createTable(soup):
    s = soup.table
    h, [_, *d] = [i.text for i in s.tr.find_all('th')], [[i.text for i in b.find_all('td')] for b in s.find_all('tr')]
    result = [dict(zip(h, i)) for i in d]

    return result

In [12]:
result = createTable(soup)

In [5]:
##Convert list of dicts to DataFrame

companies = pd.DataFrame(result)

In [6]:
##Remove extra space at end of column names

companies.columns = companies.columns.str.rstrip()

In [8]:
##Sorts the company DataFrame by the revenue column and handles the issue of 'B' and 'M'

def sortByRevenue(df):

    # Function to convert string with suffix 'B' to a numerical value
    def convert_to_number(value):
        if 'B' in value:
            return float(value.replace('B', '')) * 1e9
        if 'M' in value:
            return float(value.replace('M', '')) * 1e6
        else:
            return float(value)

    # Apply the conversion function to the 'Revenue' column
    df['Revenue_numeric'] = df['Revenue'].apply(convert_to_number)

    # Sort by the numeric 'Revenue_numeric' column
    df = df.sort_values(by='Revenue_numeric', ascending=False)

    # Drop the helper column if you don't need it
    df = df.drop(columns=['Revenue_numeric'])

    return df